# SwingSight AI GolfDB Research Notebook

This notebook is for research and model-building only. It helps you download the Kaggle golf swing videos, inspect the dataset, extract frames, test pose and detection models, explore swing phases, and export artifacts that can later connect to the local SwingSight AI dashboard.

It does not replace the app. The dashboard stays simple and coach-like, while this notebook handles the deeper experimentation.

In [ ]:
import sys
import subprocess

packages = [
    "kagglehub",
    "opencv-python",
    "ultralytics",
    "torch",
    "torchvision",
    "pandas",
    "numpy",
    "scikit-learn",
    "matplotlib",
    "tqdm",
    "pillow",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", *packages, "--no-warn-script-location"])
print("Package installation finished. Restart the kernel if your notebook asks for it.")

In [ ]:
from __future__ import annotations

import json
import math
import os
from pathlib import Path
from typing import Iterable, Optional

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib
from tqdm.auto import tqdm

def find_project_root(start: Optional[Path] = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "src").exists():
            return candidate
    return current

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

def has_video_files(folder: Path) -> bool:
    if not folder.exists():
        return False
    video_suffixes = {".mp4", ".mov", ".avi", ".mkv", ".webm"}
    return any(path.suffix.lower() in video_suffixes for path in folder.rglob("*"))

def list_video_files(folder: Path) -> list[Path]:
    video_suffixes = {".mp4", ".mov", ".avi", ".mkv", ".webm"}
    return sorted(path for path in folder.rglob("*") if path.suffix.lower() in video_suffixes)

def infer_label_hint(video_path: Path) -> str:
    parts = [part.lower() for part in video_path.parts]
    name = video_path.stem.lower()
    keywords = ["driver", "fairway", "hybrid", "iron", "wedge", "address", "backswing", "impact", "follow"]
    for keyword in keywords:
        if keyword in name or any(keyword in part for part in parts):
            return keyword
    return "unknown"

def video_metadata(video_path: Path) -> dict:
    capture = cv2.VideoCapture(str(video_path))
    fps = float(capture.get(cv2.CAP_PROP_FPS) or 0.0)
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    capture.release()
    duration_seconds = frame_count / fps if fps > 0 else None
    return {
        "video_path": str(video_path),
        "video_name": video_path.name,
        "video_id": video_path.stem,
        "parent_folder": video_path.parent.name,
        "label_hint": infer_label_hint(video_path),
        "fps": fps,
        "frame_count": frame_count,
        "width": width,
        "height": height,
        "duration_seconds": duration_seconds,
        "file_size_mb": round(video_path.stat().st_size / (1024 * 1024), 3),
    }

def preview_frame(image_path: Path, title: Optional[str] = None) -> None:
    image = cv2.imread(str(image_path))
    if image is None:
        print(f"Unable to load {image_path}")
        return
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 6))
    plt.imshow(image_rgb)
    plt.axis("off")
    plt.title(title or image_path.name)
    plt.show()

def frame_quality_metrics(image_path: Path) -> dict:
    image = cv2.imread(str(image_path))
    if image is None:
        return {"frame_path": str(image_path), "sharpness": None, "brightness": None, "contrast": None}
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    brightness = float(gray.mean())
    contrast = float(gray.std())
    return {
        "frame_path": str(image_path),
        "sharpness": sharpness,
        "brightness": brightness,
        "contrast": contrast,
    }

def extract_frames(video_path: Path, output_dir: Path, sample_fps: float = 2.0, max_frames: Optional[int] = None) -> pd.DataFrame:
    ensure_dir(output_dir)
    capture = cv2.VideoCapture(str(video_path))
    fps = float(capture.get(cv2.CAP_PROP_FPS) or 0.0)
    frame_step = max(1, int(round(fps / sample_fps))) if fps > 0 and sample_fps > 0 else 1
    saved_rows = []
    frame_index = 0
    saved_count = 0
    while True:
        success, frame = capture.read()
        if not success:
            break
        if frame_index % frame_step == 0:
            frame_name = f"{video_path.stem}_frame_{frame_index:06d}.jpg"
            frame_path = output_dir / frame_name
            cv2.imwrite(str(frame_path), frame)
            time_seconds = frame_index / fps if fps > 0 else None
            saved_rows.append({
                "video_id": video_path.stem,
                "video_path": str(video_path),
                "frame_index": frame_index,
                "time_seconds": time_seconds,
                "frame_path": str(frame_path),
            })
            saved_count += 1
            if max_frames is not None and saved_count >= max_frames:
                break
        frame_index += 1
    capture.release()
    return pd.DataFrame(saved_rows)

def safe_save_csv(df: pd.DataFrame, path: Path) -> Path:
    ensure_dir(path.parent)
    df.to_csv(path, index=False)
    return path

def normalize_keypoint_name(index: int) -> str:
    return f"kp_{index:02d}"

PROJECT_ROOT = find_project_root()
DATA_DIR = ensure_dir(PROJECT_ROOT / "data")
RAW_DIR = ensure_dir(DATA_DIR / "raw")
PROCESSED_DIR = ensure_dir(DATA_DIR / "processed")
FRAMES_DIR = ensure_dir(DATA_DIR / "frames")
SPLITS_DIR = ensure_dir(DATA_DIR / "splits")
MODELS_PRETRAINED_DIR = ensure_dir(PROJECT_ROOT / "models" / "pretrained")
MODELS_TRAINED_DIR = ensure_dir(PROJECT_ROOT / "models" / "trained")
EXPERIMENTS_DIR = ensure_dir(PROJECT_ROOT / "outputs" / "experiments")
OVERLAYS_DIR = ensure_dir(PROJECT_ROOT / "outputs" / "overlays")

print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_DIR)

## Dataset Download

This dataset is used for research, preprocessing, and model experiments only. The notebook checks for existing files first, then downloads the Kaggle dataset if needed.

If you use KaggleHub or the Kaggle API, make sure your credentials are available at `~/.kaggle/kaggle.json` or via the usual Kaggle environment variables.

In [ ]:
DATASET_SLUG = "marcmarais/videos-160"
LOCAL_DATASET_DIR = ensure_dir(RAW_DIR / "golfdb_videos_160")

def find_existing_dataset_locations() -> list[Path]:
    candidates = [LOCAL_DATASET_DIR, RAW_DIR, PROJECT_ROOT / "downloads"]
    existing = []
    for candidate in candidates:
        if has_video_files(candidate):
            existing.append(candidate)
    return existing

existing_locations = find_existing_dataset_locations()
if existing_locations:
    dataset_root = existing_locations[0]
    print(f"Using existing dataset at: {dataset_root}")
else:
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    print(f"Kaggle credentials found: {kaggle_json.exists()}")
    try:
        import kagglehub
        downloaded_path = Path(kagglehub.dataset_download(DATASET_SLUG))
        dataset_root = downloaded_path
        print(f"Downloaded dataset to: {dataset_root}")
    except Exception as exc:
        print("KaggleHub download failed. If you prefer the Kaggle API, place kaggle.json in ~/.kaggle and try again.")
        raise exc

video_files = list_video_files(dataset_root)
print(f"Video files found: {len(video_files)}")
if video_files:
    print("First few video files:")
    for path in video_files[:5]:
        print(" -", path)

## Dataset Inspection and Video Inventory

Build a clean inventory of the videos so later preprocessing, splitting, and model experiments all use the same manifest.

In [ ]:
video_inventory = pd.DataFrame([video_metadata(path) for path in video_files])
inventory_csv = PROCESSED_DIR / "golfdb_video_inventory.csv"
safe_save_csv(video_inventory, inventory_csv)
display(video_inventory.head())
print(f"Saved inventory to {inventory_csv}")

if not video_inventory.empty:
    summary = video_inventory[["fps", "frame_count", "width", "height", "duration_seconds", "file_size_mb"]].describe(include="all")
    display(summary)

## Video Preview

Look at a sample frame before extracting everything. This is a quick way to make sure the dataset is loaded correctly and the videos are readable.

In [ ]:
sample_video_path = Path(video_inventory.iloc[0]["video_path"]) if not video_inventory.empty else None
if sample_video_path is None:
    raise RuntimeError("No videos were found in the dataset.")

capture = cv2.VideoCapture(str(sample_video_path))
success, sample_frame = capture.read()
capture.release()
if not success:
    raise RuntimeError(f"Could not read the first frame from {sample_video_path}")

sample_frame_rgb = cv2.cvtColor(sample_frame, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(10, 6))
plt.imshow(sample_frame_rgb)
plt.axis("off")
plt.title(sample_video_path.name)
plt.show()

print("Sample video:", sample_video_path)
print("Metadata:", video_inventory.iloc[0].to_dict())

## Frame Extraction and Quality Checks

Extract frames at a configurable rate so you can later run pose estimation, detection, annotation, and manual labeling on a manageable image set.

In [ ]:
frames_output_dir = ensure_dir(FRAMES_DIR / sample_video_path.stem)
frame_manifest = extract_frames(sample_video_path, frames_output_dir, sample_fps=2.0, max_frames=40)
frame_manifest_csv = PROCESSED_DIR / f"{sample_video_path.stem}_frame_manifest.csv"
safe_save_csv(frame_manifest, frame_manifest_csv)
display(frame_manifest.head())
print(f"Saved frame manifest to {frame_manifest_csv}")

quality_df = pd.DataFrame([frame_quality_metrics(Path(path)) for path in frame_manifest["frame_path"].head(12)])
quality_csv = PROCESSED_DIR / f"{sample_video_path.stem}_frame_quality.csv"
safe_save_csv(quality_df, quality_csv)
display(quality_df)
print(f"Saved quality checks to {quality_csv}")

if not frame_manifest.empty:
    preview_path = Path(frame_manifest.iloc[min(3, len(frame_manifest) - 1)]["frame_path"])
    preview_frame(preview_path, title="Extracted frame preview")

## Train / Validation / Test Splits and Labeling Plan

Split by video first so that frames from the same swing do not leak across training and evaluation. If labels are limited, keep a small manually labeled set for validation before fine-tuning.

In [ ]:
split_seed = 42
split_df = video_inventory[["video_id", "video_path", "label_hint", "parent_folder"]].copy()
split_df["split"] = "train"

if len(split_df) >= 3:
    train_df, test_df = train_test_split(split_df, test_size=0.15, random_state=split_seed, shuffle=True)
    train_df, val_df = train_test_split(train_df, test_size=0.1765, random_state=split_seed, shuffle=True)
    split_df.loc[train_df.index, "split"] = "train"
    split_df.loc[val_df.index, "split"] = "val"
    split_df.loc[test_df.index, "split"] = "test"
else:
    split_df["split"] = ["train"] * len(split_df)

split_csv = SPLITS_DIR / "golfdb_video_splits.csv"
safe_save_csv(split_df, split_csv)
display(split_df.head())
print(f"Saved splits to {split_csv}")

phase_template = pd.DataFrame({
    "video_id": video_inventory["video_id"],
    "video_path": video_inventory["video_path"],
    "frame_path": [""] * len(video_inventory),
    "phase_label": ["address"] * len(video_inventory),
    "notes": ["Fill this in manually for swing phase labeling"] * len(video_inventory),
})
phase_template_csv = PROCESSED_DIR / "manual_phase_label_template.csv"
safe_save_csv(phase_template, phase_template_csv)
print(f"Saved phase label template to {phase_template_csv}")

## Pretrained YOLOv8 Pose and Club/Head Experiments

Use pretrained YOLOv8 models as a practical starting point for pose extraction and detector experimentation. These runs are for research and debugging, not final product inference.

In [ ]:
pose_overlay_dir = ensure_dir(OVERLAYS_DIR / "pose")
club_overlay_dir = ensure_dir(OVERLAYS_DIR / "club_detection")
pose_rows = []
club_rows = []
sample_frames = [Path(path) for path in frame_manifest["frame_path"].head(6)]

try:
    from ultralytics import YOLO
    pose_model = YOLO(str(MODELS_PRETRAINED_DIR / "yolov8n-pose.pt")) if (MODELS_PRETRAINED_DIR / "yolov8n-pose.pt").exists() else YOLO("yolov8n-pose.pt")
    detect_model = YOLO(str(MODELS_PRETRAINED_DIR / "yolov8n.pt")) if (MODELS_PRETRAINED_DIR / "yolov8n.pt").exists() else YOLO("yolov8n.pt")

    for frame_path in sample_frames:
        pose_results = pose_model.predict(str(frame_path), verbose=False)
        pose_result = pose_results[0]
        pose_overlay = pose_result.plot()
        pose_overlay_path = pose_overlay_dir / f"{frame_path.stem}_pose.jpg"
        cv2.imwrite(str(pose_overlay_path), pose_overlay)

        if pose_result.keypoints is not None and pose_result.keypoints.xy is not None:
            xy = pose_result.keypoints.xy.cpu().numpy()
            kp_conf = pose_result.keypoints.conf.cpu().numpy() if pose_result.keypoints.conf is not None else None
            for person_index, person_xy in enumerate(xy):
                row = {
                    "frame_path": str(frame_path),
                    "overlay_path": str(pose_overlay_path),
                    "person_index": person_index,
                }
                for kp_index, (x, y) in enumerate(person_xy):
                    row[f"{normalize_keypoint_name(kp_index)}_x"] = float(x)
                    row[f"{normalize_keypoint_name(kp_index)}_y"] = float(y)
                    if kp_conf is not None:
                        row[f"{normalize_keypoint_name(kp_index)}_conf"] = float(kp_conf[person_index, kp_index])
                pose_rows.append(row)

        detect_results = detect_model.predict(str(frame_path), verbose=False)
        detect_result = detect_results[0]
        detect_overlay = detect_result.plot()
        detect_overlay_path = club_overlay_dir / f"{frame_path.stem}_detect.jpg"
        cv2.imwrite(str(detect_overlay_path), detect_overlay)

        boxes = detect_result.boxes
        if boxes is not None:
            for box_index, box in enumerate(boxes):
                class_id = int(box.cls.item()) if hasattr(box.cls, "item") else int(box.cls)
                confidence = float(box.conf.item()) if hasattr(box.conf, "item") else float(box.conf)
                xyxy = box.xyxy[0].cpu().numpy().tolist()
                club_rows.append({
                    "frame_path": str(frame_path),
                    "overlay_path": str(detect_overlay_path),
                    "box_index": box_index,
                    "class_id": class_id,
                    "class_name": detect_result.names.get(class_id, str(class_id)) if isinstance(detect_result.names, dict) else str(class_id),
                    "confidence": confidence,
                    "x1": float(xyxy[0]),
                    "y1": float(xyxy[1]),
                    "x2": float(xyxy[2]),
                    "y2": float(xyxy[3]),
                })

except Exception as exc:
    print("YOLOv8 experiment skipped or partially failed:", exc)

pose_df = pd.DataFrame(pose_rows)
club_df = pd.DataFrame(club_rows)
pose_csv = EXPERIMENTS_DIR / "pose_landmarks_sample.csv"
club_csv = EXPERIMENTS_DIR / "club_detection_sample.csv"
if not pose_df.empty:
    safe_save_csv(pose_df, pose_csv)
    display(pose_df.head())
    print(f"Saved pose landmarks to {pose_csv}")
if not club_df.empty:
    safe_save_csv(club_df, club_csv)
    display(club_df.head())
    print(f"Saved club detection results to {club_csv}")

if sample_frames:
    preview_frame(sample_frames[0], title="Pose and detector input frame")

## Swing Phase Exploration and Feature Engineering

The dataset may not come with perfect labels. Start with weak labels, motion heuristics, and manual annotations for a smaller validation set before fine-tuning a model.

In [ ]:
def motion_score_for_sequence(frame_paths: Iterable[Path]) -> pd.DataFrame:
    previous_gray = None
    rows = []
    for index, frame_path in enumerate(frame_paths):
        image = cv2.imread(str(frame_path))
        if image is None:
            continue
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        if previous_gray is None:
            score = 0.0
        else:
            flow = cv2.calcOpticalFlowFarneback(previous_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
            score = float(np.linalg.norm(flow, axis=2).mean())
        rows.append({
            "frame_path": str(frame_path),
            "frame_index": index,
            "motion_score": score,
        })
        previous_gray = gray
    return pd.DataFrame(rows)

motion_df = motion_score_for_sequence([Path(path) for path in frame_manifest["frame_path"].head(20)])
motion_csv = EXPERIMENTS_DIR / f"{sample_video_path.stem}_motion_scores.csv"
if not motion_df.empty:
    safe_save_csv(motion_df, motion_csv)
    display(motion_df.head())
    print(f"Saved motion scores to {motion_csv}")

def build_pose_feature_table(pose_df: pd.DataFrame) -> pd.DataFrame:
    if pose_df.empty:
        return pd.DataFrame()
    feature_rows = []
    for _, row in pose_df.iterrows():
        features = {
            "frame_path": row.get("frame_path"),
            "person_index": row.get("person_index", 0),
        }
        keypoints = {}
        for key in row.index:
            if key.startswith("kp_") and key.endswith("_x"):
                base = key[:-2]
                keypoints[base] = {"x": row.get(f"{base}_x"), "y": row.get(f"{base}_y"), "conf": row.get(f"{base}_conf", np.nan)}
        if "kp_11" in keypoints and "kp_12" in keypoints:
            shoulder_width = math.dist((keypoints["kp_11"]["x"], keypoints["kp_11"]["y"]), (keypoints["kp_12"]["x"], keypoints["kp_12"]["y"]))
            features["shoulder_width"] = shoulder_width
        if "kp_23" in keypoints and "kp_24" in keypoints:
            hip_width = math.dist((keypoints["kp_23"]["x"], keypoints["kp_23"]["y"]), (keypoints["kp_24"]["x"], keypoints["kp_24"]["y"]))
            features["hip_width"] = hip_width
        if "kp_0" in keypoints and "kp_23" in keypoints and "kp_24" in keypoints:
            mid_hip_y = (keypoints["kp_23"]["y"] + keypoints["kp_24"]["y"]) / 2.0
            features["head_to_hip_y"] = keypoints["kp_0"]["y"] - mid_hip_y
        feature_rows.append(features)
    return pd.DataFrame(feature_rows)

pose_features_df = build_pose_feature_table(pose_df)
pose_features_csv = EXPERIMENTS_DIR / "pose_feature_table.csv"
if not pose_features_df.empty:
    safe_save_csv(pose_features_df, pose_features_csv)
    display(pose_features_df.head())
    print(f"Saved pose features to {pose_features_csv}")

phase_review_template = pd.DataFrame({
    "video_id": video_inventory["video_id"],
    "frame_path": [""] * len(video_inventory),
    "phase_label": ["address"] * len(video_inventory),
    "label_source": ["manual"] * len(video_inventory),
    "notes": ["Use this file to label swing phases later"] * len(video_inventory),
})
phase_review_csv = PROCESSED_DIR / "swing_phase_review_template.csv"
safe_save_csv(phase_review_template, phase_review_csv)
print(f"Saved phase review template to {phase_review_csv}")

## Baseline Model Experiment and Artifact Export

Only train once labels are ready. If labels are limited, start with a tiny manually labeled validation set and use the notebook to verify the pipeline first.

The goal is to save model weights and processed metadata that the backend can later load, while keeping the dashboard simple.

In [ ]:
labels_path = PROCESSED_DIR / "manual_swing_labels.csv"
feature_path = pose_features_csv if 'pose_features_csv' in globals() and pose_features_csv.exists() else None
training_result = {
    "trained": False,
    "reason": "No label file found yet. Create data/processed/manual_swing_labels.csv before training.",
}

if labels_path.exists() and feature_path is not None:
    labels_df = pd.read_csv(labels_path)
    features_df = pd.read_csv(feature_path)
    if "frame_path" in labels_df.columns and "frame_path" in features_df.columns:
        training_df = features_df.merge(labels_df, on="frame_path", how="inner")
    else:
        training_df = pd.DataFrame()

    if not training_df.empty and "label" in training_df.columns and training_df["label"].nunique() >= 2:
        numeric_cols = training_df.select_dtypes(include=[np.number]).columns.tolist()
        feature_cols = [col for col in numeric_cols if col not in {"label"}]
        train_set, test_set = train_test_split(training_df, test_size=0.2, random_state=42, shuffle=True, stratify=training_df["label"])
        model = RandomForestClassifier(n_estimators=200, random_state=42)
        model.fit(train_set[feature_cols], train_set["label"])
        predictions = model.predict(test_set[feature_cols])
        training_result = {
            "trained": True,
            "accuracy": float(accuracy_score(test_set["label"], predictions)),
            "feature_columns": feature_cols,
            "labels": sorted(training_df["label"].unique().tolist()),
        }
        print(classification_report(test_set["label"], predictions))
        model_path = MODELS_TRAINED_DIR / "swing_baseline_random_forest.joblib"
        joblib.dump(model, model_path)
        print(f"Saved model to {model_path}")
    else:
        training_result = {
            "trained": False,
            "reason": "Need a merged feature/label table with at least two label classes.",
        }

artifact_manifest = {
    "dataset_root": str(dataset_root),
    "inventory_csv": str(inventory_csv),
    "split_csv": str(split_csv),
    "frame_manifest_csv": str(frame_manifest_csv),
    "pose_csv": str(pose_csv) if 'pose_csv' in globals() and pose_csv.exists() else None,
    "club_csv": str(club_csv) if 'club_csv' in globals() and club_csv.exists() else None,
    "pose_features_csv": str(pose_features_csv) if 'pose_features_csv' in globals() and pose_features_csv.exists() else None,
    "training_result": training_result,
}
artifact_manifest_path = PROCESSED_DIR / "artifact_manifest.json"
artifact_manifest_path.write_text(json.dumps(artifact_manifest, indent=2), encoding="utf-8")
print(f"Saved artifact manifest to {artifact_manifest_path}")
display(pd.DataFrame([artifact_manifest]))

## Next Steps for the Local Dashboard

Use this notebook to prepare model inputs and save experiment outputs. Then connect the backend to:

- load trained weights from `models/trained/`
- read processed metadata from `data/processed/`
- read pose and overlay outputs from `outputs/experiments/` and `outputs/overlays/`
- keep the dashboard focused on simple coach-style feedback

This keeps the app local-first and beginner-friendly while still supporting advanced experimentation behind the scenes.